# 打包照片並下載（在 JetBot 上跑）

拍完照之後，**由上往下依序執行每一格**，會在格子下方出現下載連結，點一下就能存到筆電。

- 這支 notebook 要放在 `capture_samples.ipynb` 的**同一層資料夾**（`marker_calibration/`）
- 第 ① 格最上面的 `SRC` 決定打包哪個資料夾：
  - `'samples'` —— 色塊校正照
  - `'raw'` —— 工作一、三的 YOLO 照片（**連同子資料夾** `task_A_alert/`、`task_C_fall/` 一起打包）
- zip 檔名會自動加日期，例如 `raw_20261013_part1_of_3.zip`。檔名叫什麼不重要，
  **重要的是 zip 裡面保留 `raw/task_A_alert/…` 的結構**，組員解壓到 `dataset/` 底下就直接對上工作分配表
- **不需要網路**，也不需要在檔案瀏覽器按右鍵
- 程式輸出一律英文（容器 locale 不是 UTF-8，`print()` 中文會出錯）

## ① 打包

JPG、PNG 本身都已經壓縮過，所以用「不壓縮」模式（`ZIP_STORED`），速度快、檔案也不會變大。

照片太多時會自動拆成好幾個 zip，每個不超過 `MAX_MB`，避免方法 B 的瀏覽器下載失敗。
**同一個子資料夾的照片盡量放在同一個 zip**，但一個子資料夾太大時還是會被拆開。

In [ ]:
import os
import time
import zipfile

SRC = 'raw'         # 'samples' = 色塊校正照;'raw' = 工作一、三的 YOLO 照片
MAX_MB = 8          # 每個 zip 的上限
EXTS = ('.png', '.jpg', '.jpeg')

if not os.path.isdir(SRC):
    raise SystemExit('Folder %s/ not found. Check SRC, or take photos first.' % SRC)

# 連子資料夾一起找。路徑一律以 SRC 的上一層為基準,
# zip 裡才會是 raw/task_A_alert/xxx.jpg 而不是只有 xxx.jpg
base = os.path.dirname(os.path.abspath(SRC))
files = []
for root, _dirs, fnames in os.walk(SRC):
    for n in sorted(fnames):
        if n.lower().endswith(EXTS):
            full = os.path.join(root, n)
            files.append((full, os.path.relpath(os.path.abspath(full), base)))
files.sort(key=lambda f: f[1])
if not files:
    raise SystemExit('No photos under %s/ .' % SRC)

# 依累計大小分組(已依路徑排序,所以同一個子資料夾會連在一起)
groups, cur, cur_size = [], [], 0
for full, arc in files:
    size = os.path.getsize(full)
    if cur and cur_size + size > MAX_MB * 1e6:
        groups.append(cur)
        cur, cur_size = [], 0
    cur.append((full, arc))
    cur_size += size
groups.append(cur)

# 清掉這個資料夾上一次打包留下的 zip,避免下載到舊檔
tag = os.path.basename(os.path.normpath(SRC))
for old in os.listdir('.'):
    if old.startswith(tag + '_') and '_part' in old and old.endswith('.zip'):
        os.remove(old)

stamp = time.strftime('%Y%m%d')
ZIPS = []
for i, group in enumerate(groups, 1):
    zname = '%s_%s_part%d_of_%d.zip' % (tag, stamp, i, len(groups))
    with zipfile.ZipFile(zname, 'w', zipfile.ZIP_STORED) as z:
        for full, arc in group:
            z.write(full, arcname=arc)
    ZIPS.append(zname)
    print('%s  %3d photos  %.1f MB' % (zname, len(group), os.path.getsize(zname) / 1e6))

print('')
print('Total photos: %d' % len(files))
counts = {}
for _full, arc in files:
    d = os.path.dirname(arc)
    counts[d] = counts.get(d, 0) + 1
for d in sorted(counts):
    print('  %-28s %4d' % (d + '/', counts[d]))

## ② 下載：方法 A（先試這個）

點下方的連結。這是 Jupyter 內建的檔案下載路徑，不會讓 notebook 變大。

點了之後如果只是開新分頁、顯示亂碼或 404，改用方法 B。

In [ ]:
from IPython.display import HTML, display
from urllib.parse import quote

# Jupyter 的下載路徑是 /files/<相對於 Jupyter 根目錄的路徑>。
# JetBot 容器的 Jupyter 根目錄是 /workspace;若不是,改這一行。
JUPYTER_ROOT = '/workspace'

rel_dir = os.path.relpath(os.getcwd(), JUPYTER_ROOT)
links = ''.join(
    '<li><a href="/files/%s" download="%s" target="_blank">%s</a></li>'
    % (quote(os.path.join(rel_dir, z)), z, z) for z in ZIPS)
display(HTML('<b>Click each link to download:</b><ul>%s</ul>' % links))

## ② 下載：方法 B（方法 A 不行時才用）

把 zip 的內容**直接嵌進這格的輸出**裡，點連結就是從瀏覽器記憶體存檔，完全不經過 Jupyter 的檔案伺服器，所以一定下載得到。

⚠️ 代價是 notebook 檔會一起變大（每 MB 照片約多 1.4 MB）。**下載完一定要跑第 ③ 格**清掉。

In [ ]:
import base64

parts = []
for z in ZIPS:
    with open(z, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode('ascii')
    parts.append('<li><a download="%s" href="data:application/zip;base64,%s">%s</a></li>'
                 % (z, b64, z))
display(HTML('<b>Click each link to download (embedded):</b><ul>%s</ul>' % ''.join(parts)))
del parts, b64

## ③ 下載完：清理

1. 選單 **Edit → Clear All Outputs**（JupyterLab）或 **Cell → All Output → Clear**（舊版介面），再存檔。用過方法 B 的話這步**一定要做**，否則 notebook 會塞著好幾 MB 的照片
2. 執行下面這格刪掉 zip（`samples/` 裡的原始照片會保留當備份）

In [ ]:
for z in ZIPS:
    if os.path.exists(z):
        os.remove(z)
        print('Deleted', z)
print('Original photos kept in %s/' % SRC)